# IHSG vs USD/IDR — Relationship Analysis

Analysis of the daily relationship between the Jakarta Composite Index (IHSG) and the
USD/IDR exchange rate, 2019-01-01 → 2026-09-30.

**Research questions**

1. Is there a contemporaneous relationship between daily FX moves and IHSG returns?
2. Do FX returns affect IHSG with a delay (distributed lag)?
3. What is the *cumulative* effect of an FX move over the following week?
4. Which direction does causality run (FX → IHSG, IHSG → FX, or both)?

**Layout** — the logic lives in `src/ihsg_fx/`; this notebook only calls it and interprets it.
For a non-interactive run: `python analysis.py`.

## 0. Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from ihsg_fx import (  # noqa: E402
    DATA_FILE, MAX_LAG, GRANGER_MAXLAG,
    adf_table, compute_returns, drop_incomplete, get_prices,
    fit_simple_ols, fit_distributed_lag, cumulative_fx_effect, summary_table,
    residual_diagnostics, granger_both_directions,
)
from ihsg_fx import plots  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.float_format", lambda v: f"{v:,.6f}")
print("data file:", DATA_FILE)

data file: C:\Users\u7ara\Documents\Projects\IHSG\data\raw_data.csv


## 1. Choose the data window and source

Change these three values to run the whole model on a different period.

- `SOURCE = "csv"` — offline, reproducible (uses `data/raw_data.csv`).
- `SOURCE = "yfinance"` — live download for exactly this window.
- `SAVE_TO_CACHE = True` — also overwrite `data/raw_data.csv` (only for `yfinance`).

> The window is **`[START, END)`** — `END` is exclusive, matching yfinance.

In [2]:
START = "2019-01-01"
END = "2026-10-01"     # exclusive
SOURCE = "csv"         # "csv" or "yfinance"
SAVE_TO_CACHE = False

In [3]:
prices_raw = get_prices(
    source=SOURCE,
    start=START,
    end=END,
    cache_file=DATA_FILE,
    save_cache=SAVE_TO_CACHE,
)
prices = drop_incomplete(prices_raw)
ret = compute_returns(prices)

print(f"source          : {SOURCE}")
print(f"window          : {START} -> {END} (end exclusive)")
print(f"raw rows        : {len(prices_raw)}")
print(f"complete rows   : {len(prices)}")
print(f"return obs      : {len(ret)}")
prices_raw.isna().sum().to_frame("missing")

source          : csv
window          : 2019-01-01 -> 2026-10-01 (end exclusive)
raw rows        : 2018
complete rows   : 1866
return obs      : 1865


,missing
USDIDR,2
IHSG,150


## 2. Data notes

`data/raw_data.csv` holds daily closes from Yahoo Finance. Missing values occur when one
market was open and the other was closed, so we keep only days where **both** series have
an observation — the intersecting trading calendar. No interpolation is used.

### Prices

In [4]:
prices.describe().T

,count,mean,std,min,25%,50%,75%,max
USDIDR,"1,866.000000","15,266.147790","1,086.048501","13,213.240230","14,319.849850","15,009.299805","16,173.000000","18,190.300780"
IHSG,"1,866.000000","6,651.811142",825.282713,"3,937.632080","6,194.199951","6,703.748535","7,117.549316","9,134.700195"


### Returns

In [5]:
ret.describe().T

,count,mean,std,min,25%,50%,75%,max
ret_USDIDR,"1,865.000000",0.000110,0.006759,-0.042455,-0.002760,0.000134,0.002891,0.049497
ret_IHSG,"1,865.000000",-0.000010,0.011168,-0.082318,-0.005063,0.000426,0.005648,0.097042


## 2. Exploratory analysis

In [6]:
plots.plot_prices(prices);
plt.show()

C:\Users\u7ara\AppData\Local\Temp\ipykernel_15156\4188112409.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
plots.plot_return_series(ret);
plt.show()

C:\Users\u7ara\AppData\Local\Temp\ipykernel_15156\2775403305.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
plots.plot_return_scatter(ret);
plt.show()

C:\Users\u7ara\AppData\Local\Temp\ipykernel_15156\4194214582.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
plots.plot_return_distributions(ret);
plt.show()

C:\Users\u7ara\AppData\Local\Temp\ipykernel_15156\1728823426.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Reading the charts.** The levels plot shows both series trending, so any correlation on
levels is spurious — hence the analysis uses log returns. Returns are centred on zero and
strongly leptokurtic (fat tails), which is why the scatter shows only a faint negative slope.

## 3. Stationarity (ADF)

H0 = unit root (non-stationary). Rejecting H0 means the series is stationary.
Levels are expected to be non-stationary; returns should be stationary.

In [10]:
adf = adf_table({
    "IHSG (level)": prices["IHSG"],
    "USDIDR (level)": prices["USDIDR"],
    "ret_IHSG": ret["ret_IHSG"],
    "ret_USDIDR": ret["ret_USDIDR"],
})
adf

,adf_stat,p_value,used_lags,nobs,stationary
series,,,,,
IHSG (level),-1.997516,0.287663,7,1858,False
USDIDR (level),-0.844608,0.805667,6,1859,False
ret_IHSG,-15.727298,0.000000,6,1858,True
ret_USDIDR,-16.894175,0.000000,5,1859,True


## 4. Baseline model — contemporaneous only

$$r^{IHSG}_t = \alpha + \beta \, r^{USDIDR}_t + \varepsilon_t$$

Estimated by OLS with **Newey-West (HAC) standard errors** (5 lags) so inference stays
valid despite autocorrelation and heteroskedasticity.

In [11]:
baseline = fit_simple_ols(ret)
print(baseline.summary())

                            OLS Regression Results                            
Dep. Variable:               ret_IHSG   R-squared:                       0.008
Model:                            OLS   Adj. R-squared:                  0.007
Method:                 Least Squares   F-statistic:                     8.665
Date:                Thu, 29 Oct 2026   Prob (F-statistic):            0.00328
Time:                        20:36:03   Log-Likelihood:                 5744.1
No. Observations:                1865   AIC:                        -1.148e+04
Df Residuals:                    1863   BIC:                        -1.147e+04
Df Model:                           1                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const       6.437e-06      0.000      0.024      0.9

## 5. Distributed-lag model

$$r^{IHSG}_t = \alpha + \sum_{k=0}^{K} \beta_k \, r^{USDIDR}_{t-k} + \gamma \, r^{IHSG}_{t-1} + \varepsilon_t$$

Why the lags:

- FX trades nearly 24h while IDX closes around 16:00 WIB, so "daily close" pairings are
  partly misaligned.
- Information from late/overnight FX moves reaches the index on the *next* close.
- The lagged IHSG term controls for short-term momentum so the FX coefficients are not
  contaminated by own-series dynamics, and makes the reverse Granger test symmetric.
- HAC fixes *inference*, not *specification*: it protects p-values but leaves the
  point estimates unchanged if lags are omitted. That is the real reason to include them.

In [12]:
dlag = fit_distributed_lag(ret, max_lag=MAX_LAG)
print(dlag.summary())

                            OLS Regression Results                            
Dep. Variable:               ret_IHSG   R-squared:                       0.012
Model:                            OLS   Adj. R-squared:                  0.008
Method:                 Least Squares   F-statistic:                     1.944
Date:                Thu, 29 Oct 2026   Prob (F-statistic):             0.0593
Time:                        20:36:04   Log-Likelihood:                 5730.6
No. Observations:                1860   AIC:                        -1.145e+04
Df Residuals:                    1852   BIC:                        -1.140e+04
Df Model:                           7                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const       2.049e-05      0.000      0.082      0.9

In [13]:
coefs = pd.DataFrame({
    "coef": dlag.params,
    "std_err": dlag.bse,
    "p_value": dlag.pvalues,
})
coefs.loc[[i for i in coefs.index if i.startswith("USDIDR_L")]]

,coef,std_err,p_value
USDIDR_L0,-0.172562,0.054123,0.001431
USDIDR_L1,-0.081515,0.081010,0.314302
USDIDR_L2,-0.077840,0.078370,0.320598
USDIDR_L3,-0.035405,0.072865,0.627041
USDIDR_L4,0.002079,0.085413,0.980583
USDIDR_L5,0.061948,0.046919,0.186724


## 6. Cumulative effect

Individual lag coefficients are noisy and individually insignificant. What matters is the
**sum** — the total response of IHSG over the week to a 1% FX move — and the **joint**
F-test that all FX coefficients are zero together.

In [14]:
eff = cumulative_fx_effect(dlag)
pd.Series({
    "sum of FX coefficients": eff["sum_coef"],
    "SE of sum": eff["sum_se"],
    "t (sum = 0)": eff["sum_t"],
    "p (sum = 0)": eff["sum_p"],
    "joint F": eff["joint_F"],
    "joint p": eff["joint_p"],
})

sum of FX coefficients   -0.303294
SE of sum                 0.306371
t (sum = 0)              -0.989958
p (sum = 0)               0.322195
joint F                   2.212555
joint p                   0.039346
dtype: float64

In [15]:
sum_coef = eff["sum_coef"]
print(f"A 1% depreciation of IDR is associated with a cumulative "
      f"{sum_coef * 0.01 * 100:+.4f}% change in IHSG over {MAX_LAG + 1} trading days.")

A 1% depreciation of IDR is associated with a cumulative -0.3033% change in IHSG over 6 trading days.


## 7. Residual diagnostics

In [16]:
diag = pd.concat(
    [residual_diagnostics(baseline).add_prefix("baseline_"),
     residual_diagnostics(dlag).add_prefix("dlag_")],
    axis=1,
)
diag

,baseline_statistic,baseline_p_value,baseline_passed,baseline_note,dlag_statistic,dlag_p_value,dlag_passed,dlag_note
jarque_bera,"7,223.126122",0.000000,False,Jarque-Bera: normal residuals,"7,926.390670",0.000000,False,Jarque-Bera: normal residuals
durbin_watson,1.987532,NaN,True,Durbin-Watson: no serial correlation (ideal ~2),1.999581,NaN,True,Durbin-Watson: no serial correlation (ideal ~2)
breusch_pagan,0.331831,0.564583,True,Breusch-Pagan: homoskedastic residuals,131.519022,0.000000,False,Breusch-Pagan: homoskedastic residuals
ljung_box,30.187721,0.000014,False,Ljung-Box(5): no residual autocorrelation,27.990093,0.000037,False,Ljung-Box(5): no residual autocorrelation


Normality is rejected almost always for daily returns (fat tails, outliers) — that is
expected and not fatal given ~1.9k observations and HAC errors. Autocorrelation and
heteroskedasticity matter more; the HAC covariance covers both.

## 8. Model comparison

In [17]:
summary_table({"baseline": baseline, "distributed_lag": dlag})

,nobs,r2,adj_r2,aic,bic,fx_coef_t0,fx_p_t0
model,,,,,,,
baseline,1865,0.007886,0.007353,"-11,484.202201","-11,473.140169",-0.146735,0.003245
distributed_lag,1860,0.011965,0.008230,"-11,445.164668","-11,400.938014",-0.172562,0.001431


## 9. Granger causality (both directions)

In [18]:
granger = granger_both_directions(ret, "ret_IHSG", "ret_USDIDR", maxlag=GRANGER_MAXLAG)
for direction, table in granger.items():
    print(f"\n{direction}")
    print(table)


ret_USDIDR -> ret_IHSG
           F  p_value  significant
lag                               
1   0.005041 0.943408        False
2   0.965702 0.380907        False
3   0.107873 0.955516        False
4   0.106080 0.980422        False
5   0.686108 0.633984        False

ret_IHSG -> ret_USDIDR
            F  p_value  significant
lag                                
1   74.527147 0.000000         True
2   48.911944 0.000000         True
3   36.396306 0.000000         True
4   27.427730 0.000000         True
5   22.172676 0.000000         True


This is the honest answer to "does FX drive IHSG or the reverse". A significant result in
one direction means past values of that series improve the forecast of the other beyond its
own lags — it is predictive (Granger) causality, not structural causality.

## 10. Findings

In [19]:
baseline_b = baseline.params["ret_USDIDR"]
baseline_p = baseline.pvalues["ret_USDIDR"]
print(f"1. Contemporaneous beta = {baseline_b:+.4f} (p = {baseline_p:.4f}), R2 = {baseline.rsquared:.4f}")
print(f"2. Sum of FX lags      = {eff['sum_coef']:+.4f} (p = {eff['sum_p']:.4f})")
print(f"3. Joint FX F-test     = {eff['joint_F']:.3f} (p = {eff['joint_p']:.4f})")
for direction, table in granger.items():
    sig = table[table['significant']]
    print(f"4. Granger {direction}: significant at lags {list(sig.index) if len(sig) else 'none'}")

1. Contemporaneous beta = -0.1467 (p = 0.0032), R2 = 0.0079
2. Sum of FX lags      = -0.3033 (p = 0.3222)
3. Joint FX F-test     = 2.213 (p = 0.0393)


4. Granger ret_USDIDR -> ret_IHSG: significant at lags none
4. Granger ret_IHSG -> ret_USDIDR: significant at lags [1, 2, 3, 4, 5]


### Interpretation template

- **Sign convention.** USD/IDR up = rupiah weaker. A negative coefficient means a weaker
  rupiah is associated with lower IHSG returns.
- **Contemporaneous effect** is small but statistically robust; daily R² around 1% is
  normal — the FX rate is one of many drivers of the index.
- **Lag terms** are individually fragile but jointly informative; the joint F-test and the
  sum are the numbers to report, not five separate p-values.
- **Causality** can run both ways: foreign outflows hit the currency and the equity market
  simultaneously. Report Granger results as predictive evidence with a caveat.
- **Diagnostics** rejecting normality/Breusch-Pagan is expected for financial returns; the
  HAC covariance is what keeps inference credible.